In [ ]:
!pip install -q trl peft math-verify latex2sympy2_extended datasets accelerate bitsandbytes

In [ ]:
import torch

def print_vram():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    max_allocated = torch.cuda.max_memory_allocated() / 1024**3

    print(f"Allocated: {allocated:.2f} GB")
    print(f"Reserved : {reserved:.2f} GB")
    print(f"Max used : {max_allocated:.2f} GB")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from datasets import load_dataset


dataset_name = "trl-internal-testing/descriptiveness-sentiment-trl-style"
dataset = load_dataset(dataset_name, split="descriptiveness")

# Small test
dataset = dataset.select(range(200))
eval_samples = 20
train_dataset = dataset.select(range(len(dataset)-eval_samples))
eval_dataset = dataset.select(range(len(dataset)-eval_samples, len(dataset)))

print(f"Train/Eval samples: {len(train_dataset)}/{len(eval_dataset)}")


# Tokenization
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, padding_side="left")
tokenizer.add_special_tokens({"pad_token": "[PAD]"})

dataset_text_field = "prompt"

def prepare_dataset(dataset, tokenizer):
    """Tokenizza il dataset, senza pre-tokenizzazione manuale"""
    def tokenize(element):
        outputs = tokenizer(element[dataset_text_field], padding=False)
        return {"input_ids": outputs["input_ids"]}

    return dataset.map(
        tokenize,
        batched=True,
        remove_columns=dataset.column_names,
    )

train_dataset = prepare_dataset(train_dataset, tokenizer)
eval_dataset = prepare_dataset(eval_dataset, tokenizer)


# Load models
policy = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True).cuda()
ref_policy = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True).cuda()
value_model = AutoModelForSequenceClassification.from_pretrained(model_name, trust_remote_code=True, num_labels=1).cuda()
reward_model = AutoModelForSequenceClassification.from_pretrained(model_name, trust_remote_code=True, num_labels=1).cuda()


In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
from trl.experimental.ppo import PPOConfig, PPOTrainer

ppo_config = PPOConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    num_ppo_epochs=1,
    num_mini_batches=1,
    fp16=True,
    report_to=[],
)

trainer = PPOTrainer(
    args=ppo_config,
    processing_class=tokenizer,
    model=policy,
    ref_model=ref_policy,
    reward_model=reward_model,
    value_model=value_model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
)


print("VRAM prima del training:")
print_vram()

trainer.train()

print("VRAM dopo il training:")
print_vram()


output_dir = "./ppo_test_model"
trainer.save_model(output_dir)
print(f"Modello salvato in {output_dir}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/ppo_trl"

trainer.save_model(OUTPUT_DIR)
print("Modello salvato in:", OUTPUT_DIR)